# 문제해결: Advanced RAG 적용하기
## 빈칸 채우기

금융사기 보고서로 QA를 만들고, **실제 검색 청크와 답변을 읽으며** 배운 기법의 역할을 확인한다.

각 문제는 **문제 → 직접 작성 → 결과 확인 → 관찰 기록** 순서이다.
한 셀에 한 가지 작업을 넣었다. `___`(빈칸)와 `TODO`를 채운 뒤 셀을 실행한다.

### 수행 범위

| 구분 | 수행 과제 |
|---|---|
| 공통 필수 | 문서 로딩·청킹·인덱싱, Dense 기준선, 근거 기반 답변, 전후 비교·설명, Golden 최종 테스트 |
| 선택 필수 | 문제 4의 A~E 중 **한 가지** 기법 적용 |
| 선택 심화 | 다른 기법으로 한 번 더 비교 또는 새로운 질문 테스트 |

모든 기법을 실행하거나 연결하는 과제가 아니다. **문제를 확인하고 한 가지 방법을 선택하는 것**이 중요하다.  
선택 구현은 함수 정의만으로 API를 호출하지 않는다.  

문서: 하나금융연구소, 「진화하는 금융사기, 모두의 경계가 필요한 시점」, 2026.5.29.   
페이지는 표지를 1페이지로 세는 PDF 뷰어 기준이다. 인쇄된 쪽수와 1씩 다르다.

### RAG 처리 순서와 실습 순서의 차이

문서 처리는 사전에 수행하고, 질문이 들어오면 질문 처리 → 검색 → 재정렬·근거 구성 → 답변 생성을 수행한다.  
이 실습은 먼저 Dense 기준선 답변을 확인하고, 실패를 진단한 뒤 특정 단계를 개선한다.  
따라서 문제 3에서 기준선 답변을 만든 후 문제 4에서 개선하는 것은 의도한 학습 순서이다.  


## 문제 지도

| 문제 | 학생이 할 일 | 확인 질문 |
|---|---|---|
| 1 | Docling 로딩·청킹·Chroma 인덱싱 | 필요한 정보가 청크에 남아 있는가? |
| 2 | Dense로 질문 3개와 문서 밖 질문 검색 | 사람이 이 근거로 질문에 답할 수 있는가? |
| 3 | 답변 생성과 원문 대조 | 검색 오류와 답변 오류를 구분하는가? |
| 4 | 실패·취약점 진단, 기법 하나 선택 | 이 방법을 선택한 이유는 무엇인가? |
| 5 | 같은 질문으로 전후 비교 | 어떤 청크의 어떤 정보가 달라졌는가? |
| 6 | 출처·거절 확인 | 근거와 답변이 실제로 연결되는가? |
| 7 | Golden Test Set + 사람·LLM 판정 비교 | 최종 근거와 답변이 충분하고 판정도 타당한가? |
| 8 | 종합 회고와 다음 개선 제안 | 전체 실험 결과를 원문 근거로 설명할 수 있는가? |



# 1. 기존 환경 연결

기존 프로젝트의 `common_config.py`에서 `embedding_model()`, `llm_connect()`를 사용한다.
노트북은 프로젝트 루트나 그 바로 아래 폴더에서 실행하고, PDF는 노트북 폴더나 `data/`에 둔다.

필요할 때만 터미널에서 설치한다.

```bash
uv add docling "docling-core[chunking-openai]" tiktoken langchain-chroma rank-bm25 pandas ipykernel
```

LLM 연결 패키지와 API 키 설정은 기존 프로젝트를 사용한다. Docling은 첫 실행 시 모델 다운로드가 필요할 수 있다.


In [1]:
# 환경 설정: 기존 LLM·임베딩 설정 불러오기
import sys
sys.path.append("..")
from common_config import embedding_model, llm_connect

embeddings = embedding_model()
llm = llm_connect()
print("LLM과 임베딩 모델 준비 완료")

LLM과 임베딩 모델 준비 완료


In [2]:
# 기본 라이브러리 불러오기

import re
from pathlib import Path
import pandas as pd
from IPython.display import display
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi


In [3]:
# PDF가 존재하는 첫 번째 위치를 선택한다.
PDF_NAME = "진화하는 금융사기, 모두의 경계가 필요한 시점.pdf"
PDF_PATH = Path("data", PDF_NAME)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH}")

print(f"PDF: {PDF_PATH}")


PDF: data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf


## 문서를 검색 가능한 형태로 만들기

1. Docling으로 PDF를 로딩한다.
2. HybridChunker로 700토큰 기준 청킹을 한다.
3. 청크 ID·페이지를 포함한 LangChain Document로 변환한다.
4. 임베딩하여 Chroma에 인덱싱한다.
5. 원문 표7의 보상한도와 비율이 같은 청크에 있는지 확인한다.

PDF → DoclingDocument → 구조 기반 청크 → LangChain Document → 임베딩·Chroma의 흐름을 설명하라.


In [4]:
# Docling 라이브러리 불러오기
import tiktoken
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, HeadingHierarchyOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer


# 2. PDF 변환 옵션 설정

In [5]:
# ---------------------------------------------------------
# 1. PDF Pipeline 설정
# ---------------------------------------------------------
# TODO 1: 텍스트 기반 PDF이므로 OCR 사용 여부를 정하세요. (True / False)
# TODO 2: 표 구조 인식 여부를 정하세요. (True / False)
# TODO 3: PDF 제목 계층(Level) 추론을 켜세요. HeadingHierarchyOptions 사용

options = PdfPipelineOptions()
options.do_ocr = False
options.do_table_structure = True
options.heading_hierarchy_options = HeadingHierarchyOptions(
    enabled=True
)

# ---------------------------------------------------------
# 2. Docling DocumentConverter 구성
# ---------------------------------------------------------
# TODO 4: PDF 형식(InputFormat.PDF)에 위에서 만든 options를 연결하세요.
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=options
        )
    }
)
print("DocumentConverter 구성 완료")


DocumentConverter 구성 완료


# 3. HybridChunker 구성

In [6]:
# TODO 1: 임베딩 모델(EMBED_MODEL_NAME)에서 쓰는 tiktoken encoding을 불러오세요.
EMBED_MODEL_NAME = "text-embedding-3-small"
encoding = tiktoken.encoding_for_model(EMBED_MODEL_NAME)

# TODO 2: 하나의 chunk 최대 token 수를 정하세요. (문제 지시: 700 token 정도)
MAX_TOKENS = 700

# TODO 3: tiktoken encoding을 Docling tokenizer 객체로 감싸세요.
tokenizer = OpenAITokenizer(tokenizer=encoding, max_tokens=MAX_TOKENS)

# TODO 4: HybridChunker를 구성하세요.
#  - merge_peers: 같은 heading 아래의 작은 인접 chunk를 합칠지 여부
#  - repeat_table_header: 표가 나뉠 때 각 chunk에 header를 반복할지 여부
chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,
    repeat_table_header=True,
)

print("HybridChunker 구성 완료")
print(f"Embedding tokenizer : {EMBED_MODEL_NAME}")
print(f"Chunk 최대 token   : {MAX_TOKENS}")


HybridChunker 구성 완료
Embedding tokenizer : text-embedding-3-small
Chunk 최대 token   : 700


# 4. DoclingLoader + DOC_CHUNKS

In [7]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

# ---------------------------------------------------------
# DoclingLoader 구성
# ---------------------------------------------------------
# TODO 1: 원본 PDF 경로를 넣으세요. (PDF_PATH를 문자열로 변환)
# TODO 2: 앞에서 만든 converter를 넣으세요.
# TODO 3: 전체 Markdown이 아니라 Chunk 단위 Document를 받도록 export_type을 고르세요.
# TODO 4: 앞에서 만든 chunker를 넣으세요.
loader = DoclingLoader(
    file_path=str(PDF_PATH),
    converter=converter,
    export_type=ExportType.DOC_CHUNKS,
    chunker=chunker,
)


In [8]:
# ---------------------------------------------------------
# PDF → Chunked LangChain Documents
# ---------------------------------------------------------
raw_chunks = loader.load()

print(f"생성된 LangChain Document 수: {len(raw_chunks)}")


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


2026-10-07 14:55:50,939 MatchingPostProcessor WARNING  1 of 43 pdf cells matched neither a row nor a column band of the 7x4 grid and were dropped from the table
생성된 LangChain Document 수: 26


In [9]:
# 처음 몇 개 Chunk의 내용과 원본 metadata 확인
for i, doc in enumerate(raw_chunks[:3], start=1):
    print("=" * 100)
    print(f"[Chunk {i}]")
    print()
    print("[metadata]")
    print(doc.metadata)
    print()
    print("[page_content]")
    print(doc.page_content[:800])
    print()


[Chunk 1]

[metadata]
{'source': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/3', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'l': 100.38079760000001, 't': 707.086428746, 'r': 523.1917668985649, 'b': 672.028582054, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 74]}]}, {'self_ref': '#/texts/4', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'l': 109.14085, 't': 659.112697304, 'r': 523.181800297906, 'b': 626.576414896, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 76]}]}, {'self_ref': '#/texts/5', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'l': 109.14085, 't': 614.041953304, 'r': 508.79641258, 'b': 581.625539896, 'coo

# 5. Docling Metadata 정규화

Docling의 `dl_meta`는 문서 구조와 provenance를 풍부하게 담고 있지만  
그대로 Vector DB metadata에 저장하기에는 중첩 구조가 복잡하다.

따라서 실습에서는 검색·필터링·출처 표시에서 자주 사용할 값만 평탄화한다.

저장할 metadata는 다음과 같다.

| 필드 | 의미 |
|---|---|
| `source` | 원본 PDF |
| `chunk_id` | Chunk 식별 번호 |
| `page` | 대표 페이지 |
| `pages` | Chunk가 걸쳐 있는 전체 페이지 |
| `heading` | Heading breadcrumb |
| `caption` | 표/그림 Caption |
| `num_tokens` | 실제 `page_content`의 token 수 |
| `parser` | 문서 파서 |
| `chunker` | Chunking 방식 |

> `raw_chunks`는 Docling 원본 metadata를 확인할 때 사용하고,  
> `chunks`는 ChromaDB 저장에 적합하도록 metadata를 단순화한 버전으로 사용한다.

In [10]:
from langchain_core.documents import Document


def extract_pages(dl_meta: dict) -> list[int]:
    """Docling dl_meta의 provenance에서 페이지 번호를 추출한다. (제공된 함수)"""

    pages = set()

    # dl_meta["doc_items"]에는 Chunk를 구성하는 원본 문서 요소가 들어 있다.
    for item in dl_meta.get("doc_items", []):
        for prov in item.get("prov", []):
            page_no = prov.get("page_no")

            if page_no is not None:
                pages.add(page_no)

    return sorted(pages)


def normalize_docling_document(doc: Document, chunk_id: int) -> Document:
    """Docling metadata를 ChromaDB에서 다루기 쉬운 형태로 평탄화한다."""

    # DoclingLoader가 생성한 구조 metadata
    dl_meta = doc.metadata.get("dl_meta", {})

    pages = extract_pages(dl_meta)
    headings = dl_meta.get("headings") or []
    captions = dl_meta.get("captions") or []

    # page_content는 이미 HybridChunker.contextualize() 결과이므로
    # 다시 contextualize()를 호출할 필요가 없다.
    text = doc.page_content

    # TODO: 아래 metadata 필드를 채우세요. (위 markdown 표 참고)
    metadata = {
        "source": doc.metadata.get("source", ""),         # 원본 PDF 경로 (doc.metadata의 source)
        "chunk_id": chunk_id,       # Chunk 식별 번호
        "page": pages[0] if pages else -1,           # 대표 페이지 (pages가 비어 있으면 -1)
        "pages": ",".join(map(str, pages)),          # 전체 페이지를 쉼표로 이은 문자열
        "heading": " > ".join(map(str, headings)),        # heading breadcrumb (" > "로 연결)
        "caption": " | ".join(map(str, captions)),        # caption (" | "로 연결)
        "num_tokens": tokenizer.count_tokens(text),     # tokenizer로 text의 token 수 계산
        "parser": "docling",         # 문서 파서 이름
        "chunker": "HybridChunker",        # Chunking 방식 이름
    }

    return Document(
        page_content=text,
        metadata=metadata,
    )


# ---------------------------------------------------------
# Vector DB 저장용 metadata로 정규화 (제공된 코드)
# ---------------------------------------------------------
chunks = [
    normalize_docling_document(doc, chunk_id=i)
    for i, doc in enumerate(raw_chunks, start=1)
]

print(f"정규화된 LangChain Document 수: {len(chunks)}")


정규화된 LangChain Document 수: 26


## 정규화된 Metadata 확인

In [11]:
for doc in chunks[:5]:
    print("=" * 100)
    print(doc.metadata)
    print("-" * 100)
    print(doc.page_content[:600])
    print()

{'source': {'source': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/3', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'l': 100.38079760000001, 't': 707.086428746, 'r': 523.1917668985649, 'b': 672.028582054, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 74]}]}, {'self_ref': '#/texts/4', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'l': 109.14085, 't': 659.112697304, 'r': 523.181800297906, 'b': 626.576414896, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 76]}]}, {'self_ref': '#/texts/5', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'l': 109.14085, 't': 614.041953304, 'r': 508.79641258, 'b': 581.625539896, 'coord_origin':

### Chunk Token 크기확인

In [12]:
from statistics import mean, median


token_counts = [
    doc.metadata["num_tokens"]
    for doc in chunks
]

print(f"Chunk 수       : {len(token_counts)}")
print(f"최소 token     : {min(token_counts)}")
print(f"최대 token     : {max(token_counts)}")
print(f"평균 token     : {mean(token_counts):.1f}")
print(f"중앙값 token   : {median(token_counts):.1f}")


Chunk 수       : 26
최소 token     : 21
최대 token     : 791
평균 token     : 301.7
중앙값 token   : 270.5


# 6. ChromaDB 구성

In [ ]:
import shutil

from langchain_chroma import Chroma


# ---------------------------------------------------------
# DOC_CHUNKS 전용 Vector DB 경로
# ---------------------------------------------------------
DB_PATH = Path("chroma_db/financial_fraud_doc_chunks")
COLLECTION_NAME = "financial_fraud_doc_chunks"

# Chunking 설정을 수정한 뒤 다시 실습할 때는 True로 둔다.
# 이미 생성된 DB를 유지하려면 False로 변경할 수 있다.
RESET_DB = True

if RESET_DB and DB_PATH.exists():
    # 폴더를 rmtree로 지우면 이미 열린 Chroma 클라이언트가 삭제된 sqlite 파일을 잡고 있어
    # readonly database 오류가 난다. 컬렉션만 Chroma API로 삭제한다.
    try:
        Chroma(collection_name=COLLECTION_NAME, persist_directory=str(DB_PATH)).delete_collection()
        print(f"기존 컬렉션 삭제: {COLLECTION_NAME}")
    except Exception as e:
        print(f"삭제할 컬렉션 없음: {e}")

# TODO 1: 공통 설정에서 Embedding 모델을 불러오세요.
embeddings = embedding_model()
embeddings.check_embedding_ctx_length = False

# ---------------------------------------------------------
# TODO 2: Structure-aware Chunk를 ChromaDB에 저장하세요.
#  - documents: 정규화된 chunks
#  - embedding: 위에서 만든 embeddings
#  - collection_name / persist_directory: 위에서 정의한 값
#  - collection_metadata: 거리 계산 방식을 cosine으로 지정 ("hnsw:space")
# ---------------------------------------------------------
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(DB_PATH),
    collection_metadata={
        "hnsw:space": "cosine"
    },
)

print(f"저장된 Chunk 수: {vector_store._collection.count()}")


ValueError: Expected metadata value to be a str, int, float, bool, SparseVector, list, or None, got {'source': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/3', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'l': 100.38079760000001, 't': 707.086428746, 'r': 523.1917668985649, 'b': 672.028582054, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 74]}]}, {'self_ref': '#/texts/4', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'l': 109.14085, 't': 659.112697304, 'r': 523.181800297906, 'b': 626.576414896, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 76]}]}, {'self_ref': '#/texts/5', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'

### 확인 코드: 표의 항목과 수치 연결 확인

원문의 5,000만 원·85.2%·2,811억 원이 어떻게 연결되어 있는지 직접 읽는다.
문자열을 발견했다는 것만으로 표 구조 보존을 확정하지 않는다.


In [ ]:
# 아래 결과를 원문과 대조하여 해당 문제의 관찰 기록을 작성한다.
for doc in chunks:
    if "85.2%" in doc.page_content:
        print("청크:", doc.metadata["chunk_id"], "PDF 페이지:", doc.metadata["pages"])
        print(doc.page_content)


### 관찰 기록

- 원문과 청크의 보상한도·비율·배상액 연결이 일치하는가?
- 청크 ID와 페이지를 저장하는 이유는 무엇인가?
- 원문에는 있는데 청크에 없으면 검색기를 바꾸기 전에 무엇을 확인해야 하는가?
- 원문 정보가 파싱·청킹 단계에서 빠지면 검색·재정렬만으로 복구할 수 있는가? 근거를 원문 페이지로 설명하라.


## RAG 검색, 증강, 생성 문제해결
- 다음 파일로 문제해결 하기  
toy_pjt1/2-2_s_advanced_rag_financial_fraud_evaluation_student.ipynb